## 安装依赖库
安装 `transformers`、`torch` 以及加速库 `accelerate`:

In [ ]:
%pip install transformers torch accelerate

这段代码展示了在 API 高度封装之前，开发者如何通过结构化拆解来完成一次推理。它的核心思想可以总结为：将复杂的生成行为，拆解为一系列可控的任务算子（Primitives）。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 指定下载的模型
model_name = "Qwen/Qwen3-4B"

### 1. 显式实体化 (Explicit Objectification)

与 API 时代只需一个 `api_key` 不同，这里的思想是先构建工具，再执行任务。

你必须显式地将 Tokenizer（翻译官） 和 Model（大脑） 实例化。

这种逻辑延续了传统 NLP（如 HanLP）中对“组件”的依赖：你要处理文本，就必须先拥有处理文本的物理实体。

In [ ]:
# 1. 原始的加载方式，加载分词器、模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

### 2. 任务流水线化 (Pipeline Orientation)

代码中的 `pipeline("text-generation", ...)` 是典型的任务导向思想。

它明确告诉系统：我现在的任务目标是“文本生成（Text-Generation）”。

在传统 NLP 中，这可能是 `pipeline("ner")` 或 `pipeline("sentiment-analysis")`。这种思想将 LLM 视为一个大型的、可配置的任务插件。

In [ ]:
# 2. 创建推理管线
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

### 3. 面向张量的推理逻辑 (Tensor-Centric Logic)

最后一步虽然被 `generator` 简化，但其底层依然是张量（Tensor）的顺序传播。

开发者需要关心 `max_new_tokens`、`do_sample` 等控制概率分布的底层参数。

这种“微操”参数的思想，正是为了满足任务导向中对输出质量和形式的严苛要求。

In [ ]:
# 构建系统消息与用户消息
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "请用三句话介绍什么是深度学习。"},
]

# 使用聊天模板处理输入
text = tokenizer.apply_chat_template(messages,
                                     tokenize=False, add_generation_prompt=True, enable_thinking=True)

# 3. 调用模型，原始的张量推理
outputs = generator(text, max_new_tokens=100, do_sample=True, temperature=0.7)

# 打印结果
ai_message = outputs[0]["generated_text"].split("<|im_start|>assistant\n")[-1]
print(ai_message)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Device set to use cuda:0


<think>
好的，用户让我用三句话介绍深度学习。首先，我需要明确深度学习的定义，它属于机器学习的一个子领域，使用深层神经网络。接下来要说明其核心特点，比如模拟人脑处理数据，通过多层结构自动提取特征。然后要提到应用领域，如图像识别、自然语言处理等，以及它的优势，比如处理复杂模式的能力。还要确保语言简洁专业，避免术语过多，让不同背景的用户都能理解
